In [1]:
import napari
import tifffile as tiff

HE_PATH = "data/H&E/Project_BLOCCO_1_ch00.tif"

ARVIS = {
  "c26STAT3": {
      "path": "data/ARVIS/arivis_blocco1/blocco1_c26STAT3_finalprediction.tiff",
      "height": 8000,
      "y_offset": 0,
  },
  "sham": {
      "path": "data/ARVIS/arivis_blocco1/blocco1_sham_finalprediction.tiff",
      "height": 7000,
      "y_offset": 8000,
  },
  "c26foxO": {
      "path": "data/ARVIS/arivis_blocco1/blocco1_c26foxO_finalprediction.tiff",
      "height": 7718,
      "y_offset": 15000,
  },
}

he = tiff.imread(HE_PATH)

viewer = napari.Viewer()
viewer.add_image(
  he,
  name="H&E blocco1",
  rgb=True,
)

for sample, info in ARVIS.items():
  labels = tiff.imread(info["path"])

  # Crop to match H&E/json dimensions:
  # width 16166, sample-specific height.
  labels = labels[: info["height"], :16166]

  viewer.add_labels(
      labels,
      name=f"ARVIS {sample}",
      opacity=0.45,
      blending="translucent",
      translate=(info["y_offset"], 0),  # napari uses (y, x)
  )

napari.run()

In [2]:
import napari
import tifffile
import geopandas as gpd
import numpy as np

tiff_path = "data/H&E/Project_BLOCCO_1_ch00.tif"
geojson_path = "data/ARVIS/Project_Blocco_1_ch00.geojson"

# carica immagine
img = tifffile.imread(tiff_path)

# carica geojson
gdf = gpd.read_file(geojson_path)

# converti geometrie GeoJSON in lista di poligoni napari
shapes = []
for geom in gdf.geometry:
    if geom is None:
        continue

    if geom.geom_type == "Polygon":
        coords = np.asarray(geom.exterior.coords)
        # shapely/geopandas usa x,y; napari vuole y,x
        shapes.append(coords[:, [1, 0]])

    elif geom.geom_type == "MultiPolygon":
        for poly in geom.geoms:
            coords = np.asarray(poly.exterior.coords)
            shapes.append(coords[:, [1, 0]])

viewer = napari.Viewer()
viewer.add_image(img, name="TIFF")

viewer.add_shapes(
    shapes,
    shape_type="polygon",
    name="GeoJSON",
    edge_width=1,
    face_color="transparent",
)

napari.run()

Traceback (most recent call last):
  File "/Users/riccardo/micromamba/envs/sc/lib/python3.11/site-packages/vispy/app/backends/_qt.py", line 699, in event
    self._handle_native_gesture_event(ev)
  File "/Users/riccardo/micromamba/envs/sc/lib/python3.11/site-packages/vispy/app/backends/_qt.py", line 640, in _handle_native_gesture_event
    vispy_event = self._vispy_canvas.events.touch(
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/riccardo/micromamba/envs/sc/lib/python3.11/site-packages/vispy/util/event.py", line 453, in __call__
    self._invoke_callback(cb, event)
  File "/Users/riccardo/micromamba/envs/sc/lib/python3.11/site-packages/vispy/util/event.py", line 471, in _invoke_callback
    _handle_exception(self.ignore_callback_errors,
  File "/Users/riccardo/micromamba/envs/sc/lib/python3.11/site-packages/vispy/util/event.py", line 469, in _invoke_callback
    cb(event)
  File "/Users/riccardo/micromamba/envs/sc/lib/python3.11/site-packages/napari/_vispy/canvas.p